# Hướng Dẫn Toàn Diện Mã Hóa SAT: AMK, ALK, Ladder Encoding và Nurse Rostering
Notebook này cung cấp hướng dẫn từng bước đi kèm mã nguồn thực thi chi tiết cho các dạng ràng buộc Cardinality trong SAT Solver sử dụng thư viện `PySAT`.

In [ ]:
!pip install python-sat

## 1. At Most K (AMK) - Sequential Counter Encoding
Ràng buộc AMK đảm bảo rằng tối đa chỉ có $K$ biến trong tập hợp $N$ biến nhận giá trị `True`. Phương pháp `Sequential Counter` xây dựng một ma trận biến phụ $s_{i,j}$ đóng vai trò như bộ đếm tích lũy trạng thái tiền tố.

In [ ]:
from pysat.solvers import Minisat22

def encode_amk_sc(variables, k, start_aux):
    n = len(variables)
    if k >= n: return [], start_aux, []
    if k == 0: return [[-v] for v in variables], start_aux, []

    clauses = []
    s = {}
    aux_id = start_aux

    for i in range(1, n):
        for j in range(1, k + 1):
            s[(i, j)] = aux_id
            aux_id += 1

    clauses.append([-variables[0], s[(1, 1)]])
    for j in range(2, k + 1):
        clauses.append([-s[(1, j)]])

    for i in range(1, n - 1):
        xi = variables[i]
        clauses.append([-xi, s[(i + 1, 1)]])
        clauses.append([-s[(i, 1)], s[(i + 1, 1)]])
        
        for j in range(2, min(i + 1, k) + 1):
            clauses.append([-xi, -s[(i, j - 1)], s[(i + 1, j)]])
            clauses.append([-s[(i, j)], s[(i + 1, j)]])
            
        clauses.append([-xi, -s[(i, k)]])

    clauses.append([-variables[n - 1], -s[(n - 1, k)]])
    output_states = [s[(n - 1, j)] for j in range(1, k + 1)] if n > 1 else []
    
    return clauses, aux_id, output_states

# Kiểm tra lỗi điều kiện AMK (Chọn tối đa 2 trong 4 biến)
solver = Minisat22()
vars_list = [1, 2, 3, 4]
clauses, _, _ = encode_amk_sc(vars_list, k=2, start_aux=5)
for c in clauses: solver.add_clause(c)

# Ép 3 biến bằng True để thử nghiệm
solver.add_clause([1])
solver.add_clause([2])
solver.add_clause([3])
print("Kết quả kiểm tra AMK (Mong đợi: False):", solver.solve())
solver.delete()

## 2. At Least K (ALK) - Sequential Counter Encoding
Ràng buộc ALK yêu cầu có ít nhất $K$ biến trong tập mang giá trị `True`. Bản chất tương đương lôgic: Tối đa có $N-K$ biến mang giá trị `False`. Ta tiến hành phủ định toàn bộ biến đầu vào và áp dụng hàm AMK.

In [ ]:
def encode_alk_sc(variables, k, start_aux):
    n = len(variables)
    max_false = n - k
    negated_vars = [-v for v in variables]
    return encode_amk_sc(negated_vars, max_false, start_aux)

# Kiểm tra lỗi điều kiện ALK (Ít nhất 2 trong 4 biến đúng)
solver = Minisat22()
vars_list = [1, 2, 3, 4]
clauses, _, _ = encode_alk_sc(vars_list, k=2, start_aux=5)
for c in clauses: solver.add_clause(c)

# Ép 3 biến nhận giá trị False
solver.add_clause([-1])
solver.add_clause([-2])
solver.add_clause([-3])
print("Kết quả kiểm tra ALK (Mong đợi: False):", solver.solve())
solver.delete()

## 3. Ladder AMK Encoding
Ladder Encoding tối ưu hóa việc biểu diễn hai cửa sổ trượt giao nhau bằng cách dùng chung khối Sequential Counter ở giữa (`shared_block`), và liên kết phần tử biên `x_start` và `x_end` thông qua mệnh đề lôgic phụ thuộc biến đếm.

In [ ]:
def encode_ladder_step(x_start, shared_block, x_end, k, start_aux):
    clauses = []
    block_clauses, next_aux, shared_out_states = encode_amk_sc(shared_block, k, start_aux)
    clauses.extend(block_clauses)
    
    if len(shared_out_states) >= k:
        s_k = shared_out_states[k - 1]
        clauses.append([-x_start, -s_k])
        clauses.append([-x_end, -s_k])
        
    if len(shared_out_states) > k:
        s_k_plus_1 = shared_out_states[k]
        clauses.append([-s_k_plus_1])
        
    return clauses, next_aux

def encode_sequence_alsc_ladder(variables, w, k, start_aux):
    clauses = []
    n = len(variables)
    max_false = w - k
    aux_id = start_aux
    neg_vars = [-v for v in variables]
    
    for i in range(n - w):
        x_start = neg_vars[i]
        shared_block = neg_vars[i + 1 : i + w]
        x_end = neg_vars[i + w]
        
        step_clauses, aux_id = encode_ladder_step(x_start, shared_block, x_end, max_false, aux_id)
        clauses.extend(step_clauses)
        
    return clauses, aux_id

## 4. Mô hình hóa hoàn chỉnh Nurse Rostering Problem (NRP)
Bài toán mở rộng cho $N$ y tá (`num_nurses`) làm việc trên chuỗi ngày `num_days`. Mỗi cửa sổ trượt độ dài $W$ ngày liên tiếp yêu cầu y tá phải trực ít nhất $K$ ca làm việc.

In [ ]:
def solve_nurse_rostering(num_nurses, num_days, w, k):
    solver = Minisat22()
    aux_id = num_nurses * num_days + 1
    
    # Tạo ma trận biến ID nguyên dương cho từng cặp (Y tá, Ngày)
    nurse_vars = {}
    var_counter = 1
    for n in range(num_nurses):
        nurse_vars[n] = []
        for d in range(num_days):
            nurse_vars[n].append(var_counter)
            var_counter += 1
            
    # Áp dụng ràng buộc chuỗi Ladder cho từng y tá độc lập
    for n in range(num_nurses):
        clauses, aux_id = encode_sequence_alsc_ladder(nurse_vars[n], w, k, aux_id)
        for clause in clauses: 
            solver.add_clause(clause)
            
    is_sat = solver.solve()
    print(f"Trạng thái bài toán NRP (Nurses={num_nurses}, Days={num_days}, W={w}, K={k}):", is_sat)
    
    if is_sat:
        model = solver.get_model()
        for n in range(num_nurses):
            schedule = []
            for d in range(num_days):
                day_var = nurse_vars[n][d]
                schedule.append("X" if day_var in model else "-")
            print(f"Lịch y tá {n+1}: {' '.join(schedule)}")
            
    solver.delete()

# Thực thi chạy thử nghiệm hệ thống xếp lịch
solve_nurse_rostering(num_nurses=3, num_days=10, w=4, k=2)